Goal of this notebook is to teach how to train a neural network to predict the next legal move for a given game state. You do not need to know the basics of the architecture yet, so the description remains at a high level.

In [ ]:
import os 
import sys 
main_dir = os.path.dirname(os.getcwd())
sys.path.append(main_dir)
from src.game_utils import run_models_against_each_other, run_random_game
from src.train_game_utils import define_model, train_model
from tqdm import tqdm

In [ ]:
# First step, generate training data using random game
num_games_training = 500 
training_data = []
for _ in tqdm(range(num_games_training), desc="Generating training data"):
    game_data = run_random_game()
    training_data.append(game_data['all_user_inputs'][1:]) # Exclude the first move which is always the same


In [ ]:
# Show how the training data looks like for one game
import pandas as pd
from tabulate import tabulate
print("Example game data:")
df = pd.DataFrame(training_data[0], columns=['board_state', 'player', 'next_move'])
print(tabulate(df, headers='keys', tablefmt='psql'))



In [ ]:
# What we need for training data is the current board state as input and the next move as the target
tictactoe_model = define_model(reproducible=True)
# Prepare the training data
X_train = []
y_train = []
for game in training_data:
    for move in game:
        board_state, player, next_move = move
        X_train.append(board_state)
        y_train.append(next_move)

# Train the model
tictactoe_model = train_model(tictactoe_model, X_train, y_train, num_epochs=10)


In [ ]:
# Did the model learn anything? Let's test it against a random player to see if it predicts an illegal move
from src.game_utils import test_model_for_illegal_moves, show_game_sequence

game_data = test_model_for_illegal_moves(tictactoe_model, verbose=True)


Was it all legal moves? Check automatically for a large number of games and see what percentage of games were legal. 

In [ ]:
# Test it for many games to see how it performs
number_of_games_with_illegal_moves = 0
num_test_games = 10000
for _ in tqdm(range(num_test_games), desc="Testing model for illegal moves"):
    game_data = test_model_for_illegal_moves(tictactoe_model, verbose=False)
    if game_data['illegal_moves']:
        number_of_games_with_illegal_moves += 1

print(f"Model made illegal moves in {number_of_games_with_illegal_moves} out of {num_test_games} games.")
print(f"Percentage of games with illegal moves: {number_of_games_with_illegal_moves / num_test_games * 100:.2f}%")

Not happy with the training? Increase the number of training data, or the number of epochs and see if that helps you get a better model. 

In [ ]:
# Save the model to a file so we can load it later without retraining
save = False
if save:
    import torch
    import datetime
    model_save_folder = os.path.join(main_dir, 'src', 'models')
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    model_save_path = os.path.join(model_save_folder, f'tictactoe_model_{timestamp}.pt')
    torch.save(tictactoe_model.state_dict(), model_save_path)
    print(f"Model saved to {model_save_path}")